In [ ]:
import pandas as pd
import requests
from datetime import datetime , timedelta
import os
from mecoda_minka import get_obs, get_dfs
import folium
from folium.plugins import HeatMap
from html2image import Html2Image

API_PATH = "https://api.minka-sdg.org/v1"

# 1. Construir un dataframe de métricas principales del proyecto y variación en el último mes.

El dataframe resultante tendrá esta forma:

```python
metric, number_today, number_in_last_month
observations, 1521, 23
observers, 124, 2
identifiers, 26, 0
species, 462, 5
```

* number_today = dato actual de observaciones, observadores, identificadores, especies.
* number_in_last_month = dato de fecha actual menos dato registrado 30 días antes (variación en los últimos 30 días).

Usamos llamadas a la API, porque son datos totales. 

Creamos un directorio "data" donde guardaremos todos los csv que vamos a generar. Este dataframe lo guardamos como "data/main_metrics.cvs", sin incluir los índices (index=False).


In [ ]:
# -----> Corregir esta función: dato fecha de hoy y 30 días antes, nombres de columnas correctos

def get_main_metrics(id_project, grade=None):

    """
    Obtiene las métricas principales de un proyecto.
    :param id_project: ID del proyecto
    :return: DataFrame con las métricas principales
    """
    
    # Primero he definido las fechas que se utilizaran para obtener las metricas : Defino el mes actual y el mes anterior

    # MES ACTUAL
    today = datetime.today()
   
    # MES ANTERIOR
    last_month_date = today - timedelta(days=30)


    base_url = 'https://api.minka-sdg.org/v1'

  
    # METRICAS PARA EL MES ANTERIOR
    url_observations_last = f'{base_url}/observations?project_id={id_project}&month={last_month}&quality_grade=research'
    observations_results_last = requests.get(url_observations_last).json()['total_results']

    url_observers_last = f'{base_url}/observations/observers?project_id={id_project}&month={last_month}&quality_grade=research'
    observers_results_last = requests.get(url_observers_last).json()['total_results']

    url_identifiers_last = f'{base_url}/observations/identifiers?project_id={id_project}&month={last_month}&quality_grade=research'
    identifiers_results_last = requests.get(url_identifiers_last).json()['total_results']

    url_species_last = f'{base_url}/observations/species_counts?project_id={id_project}&month={last_month}&quality_grade=research'
    species_results_last = requests.get(url_species_last).json()['total_results']

    
    # METRICAS PARA EL MES ACTUAL 
    url_observations_today = f'{base_url}/observations?project_id={id_project}&month={this_month}'
    observations_results_today = requests.get(url_observations_today).json()['total_results']

    url_observers_today = f'{base_url}/observations/observers?project_id={id_project}&month={this_month}&quality_grade=research'
    observers_results_today = requests.get(url_observers_today).json()['total_results']

    url_identifiers_today = f'{base_url}/observations/identifiers?project_id={id_project}&month={this_month}&quality_grade=research'
    identifiers_results_today = requests.get(url_identifiers_today).json()['total_results']

    url_species_today = f'{base_url}/observations/species_counts?project_id={id_project}&month={this_month}&quality_grade=research'
    species_results_today = requests.get(url_species_today).json()['total_results']


    # Aqui genero el dataframe para almacenar los datos (TODOS LOS DATOS SON CON EL RESEARCH GRADE)

    df_main_metrics = pd.DataFrame({
        'Metric': ['Observations', 'Observers', 'Identifiers', 'Species'],
        'Last Month': [observations_results_last, observers_results_last, identifiers_results_last, species_results_last],
        'Current Month': [observations_results_today, observers_results_today, identifiers_results_today, species_results_today]
    })

    os.makedirs('data', exist_ok=True)
    df_main_metrics.to_csv(f'data/main_metrics_{id_project}.csv', index=False)

    return df_main_metrics

In [ ]:
df_main_metrics = get_main_metrics(264)

# 2. Evolución de las métricas principales

Construir un dataframe con esta forma:

```python
month, observations, observers, identifiers, species
2024-01, 185, 15, 7, 62
2024-02, 128, 3, 1, 32
...
```

Los datos no son acumulativos, son del mes en concreto. Lo sacaremos usando llamadas a la API.

Para ello puedes utilizar estas funciones, que te ayudarán a construirlo:

In [ ]:
def get_totals(project_id, year, month, kind="project", session=None):

    if session is None:
        session = requests.Session()

    # Define the API endpoints for the different metrics
    # and construct the URLs with the provided project ID and date range
    if kind == "project":
        url_obs = f"{API_PATH}/observations?project_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?project_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?project_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?project_id={project_id}&month={month}&year={year}"
    elif kind == "place":
        url_obs = f"{API_PATH}/observations?place_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?place_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?place_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?place_id={project_id}&month={month}&year={year}"  

    # Make GET requests to the API endpoints and extract the total results
    total_obs = session.get(url_obs).json()["total_results"]
    total_part = session.get(url_part).json()["total_results"]
    total_ident = session.get(url_ident).json()["total_results"]
    total_spe = session.get(url_spe).json()["total_results"]

    return total_obs, total_part, total_ident, total_spe

In [ ]:
from datetime import datetime
import calendar

def get_month_list(years: list) -> dict:
    current_year = datetime.now().year
    current_month = datetime.now().month
    meses = []

    for year in years:
        max_month = 12 if year < current_year else current_month
        for month in range(1, max_month + 1):
            meses.append(f"{year}-{str(month).zfill(2)}")
            
    return meses

In [ ]:
meses = get_month_list(range(2022, datetime.now().year + 1))
meses

In [ ]:
# Para cada elemento de la lista, podemos sacar el año y el mes
meses[0].split("-")[0]  # Año
meses[0].split("-")[1]  # Mes

In [ ]:
# Ejemplo de uso con el primer mes

get_totals(
    project_id=264,
    year=int(meses[0].split("-")[0]),
    month=int(meses[0].split("-")[1]),
    kind="project"
)

Esto nos devuelve un diccionario con los meses como clave y el último día del mes como valor. Así podemos usarlo con la función anterior:

Ahora hay que unir las dos funciones para sacar cada mes y de cada mes sacar los valores de las métricas. Eso nos da los resultados de un mes, que podemos guardar en un diccionario. Y luego unimos los diccionarios de cada mes en una lista de todos los meses. Te pongo debajo un ejemplo de uso con un mes.

In [ ]:
# Ejemplo de proceso con un mes

# Creamos una lista vacía para almacenar los resultados de cada mes
total_metrics = []

# Te enseño cómo construirlo con el primer mes de la lista meses
mes = meses[0]
year = mes.split("-")[0]
month = mes.split("-")[1]
total_obs, total_spe, total_part, total_ident = get_totals(
    project_id=264, year=year, month=month, kind="project"
)

# Creamos un diccionario con los resultados del mes
# y lo añadimos a la lista de métricas totales
total_for_month = {}
total_for_month["month"] = meses[0]
total_for_month["total_obs"] = total_obs
total_for_month["total_spe"] = total_spe
total_for_month["total_part"] = total_part
total_for_month["total_ident"] = total_ident
total_metrics.append(total_for_month)

# Creamos un DataFrame a partir de la lista de métricas totales
df_monthly = pd.DataFrame(total_metrics)

Ahora hay que crear una función que itere por todos los elementos de la lista meses desde el inicio de MINKA y saque los datos para cada mes usando get_totals a un diccionario, los acumule en la lista y la lista la convierta a un dataframe.

Es decir, para cada elemento de los meses, usamos get_totals para sacar las métricas y las almacenamos. Si lo ves complicado lo hacemos juntos.

Ese dataframe lo guardamos como "data/monthly_metrics.csv".

In [ ]:
# Aqui he copiado y pegado tal cual el script de ejemplo con la finalidad de obtener las requests a la API

API_PATH = "https://api.minka-sdg.org/v1"

def get_totals(project_id, year, month, kind="project", session=None):
    if session is None:
        session = requests.Session()

    if kind == "project":
        url_obs = f"{API_PATH}/observations?project_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?project_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?project_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?project_id={project_id}&month={month}&year={year}"
    elif kind == "place":
        url_obs = f"{API_PATH}/observations?place_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?place_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?place_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?place_id={project_id}&month={month}&year={year}"

    total_obs = session.get(url_obs).json().get("total_results", 0)
    total_part = session.get(url_part).json().get("total_results", 0)
    total_ident = session.get(url_ident).json().get("total_results", 0)
    total_spe = session.get(url_spe).json().get("total_results", 0)

    return total_obs, total_part, total_ident, total_spe

In [ ]:
# Aqui se crea la lista de meses tal y como se propone en  el ejemplo

def get_month_list(years: list) -> list:
    current_year = datetime.now().year
    current_month = datetime.now().month
    meses = []

    for year in years:
        max_month = 12 if year < current_year else current_month
        for month in range(1, max_month + 1):
            meses.append(f"{year}-{str(month).zfill(2)}")
    return meses

In [ ]:
# En esta casilla se genera la funcion que recorre los meses y obtiene las metricas de cada uno de ellos

def build_monthly_metrics(project_id):
    meses = get_month_list(range(2022, datetime.now().year + 1))
    total_metrics = []

    session = requests.Session()

    for mes in meses:
        year = int(mes.split("-")[0])
        month = int(mes.split("-")[1])

        # Agrego un print para saber si la evolución de los meses se esta ejecutando de forma correcta 

        print(f"Procesando {mes}...")

        # Aqui me ayudo ChatGPT, diciendo que estas lineas eran utiles para que si hay algun error no rompa todo el codigo y se almacene en la variable e

        try:
            total_obs, total_part, total_ident, total_spe = get_totals(
                project_id=project_id, year=year, month=month, kind="project", session=session
            )
        except Exception as e:
            print(f"Error procesando {mes}: {e}")
            continue


        # Generamos el diccionario para posteriormente añadirlos a la lista total_metrics 
        total_for_month = {
            "month": mes,
            "observations": total_obs,
            "observers": total_part,
            "identifiers": total_ident,
            "species": total_spe
        }

        total_metrics.append(total_for_month)

    df_monthly = pd.DataFrame(total_metrics)


    os.makedirs("data", exist_ok=True)
    df_monthly.to_csv("data/monthly_metrics.csv", index=False)

    return df_monthly

In [ ]:
df_monthly = build_monthly_metrics(264)

In [7]:
def get_total_dataframes(id_project, grade):
    observations = get_obs(id_project=id_project, grade=grade)
    df_obs, df_photos = get_dfs(observations)

    os.makedirs("data", exist_ok=True)
    df_obs.to_csv("data/observations.csv", index=False)
    df_photos.to_csv("data/photos.csv", index=False)

In [17]:
observations = get_obs(id_project=264, grade="research")
df_obs, df_photos = get_dfs(observations)

Generating list of observations:
https://api.minka-sdg.org/v1/observations?project_id=264&quality_grade=research&per_page=200
Total observations to download: 4556
Number of elements: 200
Number of elements: 400
Number of elements: 600
Number of elements: 800
Number of elements: 1000
Number of elements: 1200
Number of elements: 1400
Number of elements: 1600
Number of elements: 1800
Number of elements: 2000
Number of elements: 2200
Number of elements: 2400
Number of elements: 2600
Number of elements: 2800
Number of elements: 3000
Number of elements: 3200
Number of elements: 3400
Number of elements: 3600
Number of elements: 3800
Number of elements: 4000
Number of elements: 4200
Number of elements: 4400
Number of elements: 4556


# 3. Taxonomías

Descargamos todas las observaciones del proyecto, usando mecoda_minka. Generamos los dataframes de observaciones y de fotos. A partir de df_obs creamos una función que nos permita ver el número de observaciones por reino, filo, clase... Este rango se tiene que poder indicar como parámetro, para usar la misma función para cualquier rango.

Guardamos df_obs y df_photos como csv en la carpeta `data`. Y no guardamos los índices, como en los casos anteriores.

Creamos la función. Ten en cuenta las columnas de los rangos que tiene el dataframe de df_obs. En las columnas "kingdom", "phylum", "class",... están los rangos taxonómicos superiores a la identificación. El "taxon_name" es el identificado en la observación, y el "taxon_rank" el rango de la identificación. En las otras columnas están los rangos superiores. Esto lo hacemos juntos, que es un poco complicado de explicar por escrito.

In [ ]:
def get_taxon_count(df_obs, rank_level):
    """
    Obtiene el conteo de taxones por nivel de rango.
    :param df_obs: DataFrame con las observaciones
    :param rank_level: Nivel de rango (kingdom, phylum, class, order, family, genus, species)
    :return: DataFrame con el conteo de taxones
    """

    if rank_level not in df_obs.columns:
        raise ValueError(f"'{rank_level}' esta columna no existe en el DataFrame.")

    df_taxon_counts = df_obs[rank_level].value_counts().reset_index()
    df_taxon_counts.columns = [rank_level, "observation_count"]

    return df_taxon_counts

In [20]:
get_taxon_count(df_obs, "kingdom")

,kingdom,observation_count
0,Animalia,3106
1,Plantae,1320
2,Chromista,115
3,Fungi,7


In [ ]:
get_taxon_count(df_obs, "phylum")

In [ ]:
get_taxon_count(df_obs, "class")

In [5]:
df_obs.head()

,id,created_at,updated_at,observed_on,observed_on_time,iconic_taxon,taxon_id,taxon_rank,taxon_name,latitude,...,identifiers,num_identification_agreements,num_identification_disagreements,device,kingdom,phylum,class,order,family,genus
0,450052,2025-05-03,2025-05-05,2025-04-08,13:43:00,plantae,46253,genus,Matthiola,41.418353,...,"crismc, xasalva, crismc",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Capparales,Brassicaceae,None
1,450051,2025-05-03,2025-05-04,2025-04-08,13:52:00,aves,250485,species,Sturnus vulgaris,41.418366,...,"crismc, xasalva",1,0,web,Animalia,Chordata,Aves,Passeriformes,Sturnidae,Sturnus
2,450050,2025-05-03,2025-05-05,2025-04-08,13:41:00,plantae,246170,species,Medicago marina,41.418362,...,"crismc, xasalva",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Fabales,Fabaceae,Medicago
3,450049,2025-05-03,2025-05-05,2025-04-08,13:40:00,plantae,40121,genus,Malva,41.418767,...,"crismc, xasalva",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Malvales,Malvaceae,None
4,450048,2025-05-03,2025-05-05,2025-04-08,13:39:00,insecta,252648,species,Oxythyrea funesta,41.418766,...,"crismc, xasalva",1,0,web,Animalia,Arthropoda,Insecta,Coleoptera,Cetoniidae,Oxythyrea


In [6]:
# Función modificada

def get_taxon_count(df_obs, rank_level):
    """
    Obtiene el conteo de taxones por nivel de rango.
    :param df_obs: DataFrame con las observaciones
    :param rank_level: Nivel de rango (kingdom, phylum, class, order, family, genus, species)
    :return: DataFrame con el conteo de taxones
    """

    if rank_level not in df_obs.columns:
        if rank_level == "species":
            df_taxon_counts = df_obs.loc[df_obs['taxon_rank'] == rank_level, "taxon_name"].value_counts().reset_index()
        else:
            raise ValueError(f"'{rank_level}' esta columna no existe en el DataFrame.")

    if rank_level in df_obs.columns:
        df_taxon_counts = df_obs[rank_level].value_counts().reset_index()
        df_taxon_counts.columns = ["taxon_name", "count"]

    # Si hay observaciones identificadas como este rango
    if rank_level != "species":
        df2 = df_obs.loc[df_obs['taxon_rank'] == rank_level, "taxon_name"].value_counts().reset_index()

        if len(df2) > 0:

            # Concatenamos ambos DataFrames
            df_combined = pd.concat([df_taxon_counts, df2])

            # Agrupamos por taxon_name y sumamos los counts
            df_summed = df_combined.groupby('taxon_name', as_index=False)['count'].sum()
            
            df_sorted = df_summed.sort_values(by='count', ascending=False).reset_index(drop=True)
            
        else:
            df_sorted = df_taxon_counts
    else:
        df_sorted = df_taxon_counts

    df_sorted['taxon_rank'] = rank_level

    return df_sorted[['taxon_rank', 'taxon_name', 'count']]

In [8]:
for rank_level in ['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']:
    print(get_taxon_count(df_obs, rank_level))
    print()

  taxon_rank taxon_name  count
0    kingdom   Animalia   3108
1    kingdom    Plantae   1326
2    kingdom  Chromista    115
3    kingdom      Fungi      7

   taxon_rank     taxon_name  count
0      phylum   Tracheophyta   1211
1      phylum       Mollusca    865
2      phylum       Chordata    856
3      phylum     Arthropoda    843
4      phylum       Cnidaria    325
5      phylum  Echinodermata    158
6      phylum     Ochrophyta    112
7      phylum     Rhodophyta     96
8      phylum        Bryozoa     38
9      phylum    Chlorophyta     19
10     phylum       Annelida     16
11     phylum       Porifera      7
12     phylum  Basidiomycota      6
13     phylum       Radiozoa      3
14     phylum     Zygomycota      1

   taxon_rank        taxon_name  count
0       class     Magnoliopsida    865
1       class              Aves    742
2       class           Insecta    662
3       class          Bivalvia    517
4       class        Liliopsida    337
5       class        Gastropoda  

# 4. Especies vistas por primera vez en el proyecto desde el último informe (últimos 30 días)

A partir del df_obs podemos sacar este dato fácilmente. Toma el dataframe, ordénalo por fecha de observación, en orden ascendente (las primeras observaciones estarán más arriba). Ahora quédate solo con las primeras observaciones de cada especie. Es decir:
* Seleccionamos aquellas observaciones que hayan llegado al nivel de especie (columna "taxon_rank" == "species").
* Nos quedamos con la primera observación de cada especie, usando drop_duplicates()
```python
df_first = df_obs.drop_duplicates(subset=["taxon_name"], keep="first")
```
* Así nos quedaremos con la primera observación de cada especie. Ahora filtramos de esta tabla las que tengan fecha de observación mayor a hoy menos 30 días (vistas en los últimos 30 días).

Esas serán las especies nuevas observadas en los últimos 30 días.

Primero haz el proceso y luego lo conviertes a una función. Es decir, carga el df_obs y haz los pasos con él, cuando te haya salido ya lo conviertes en función.

In [18]:
def get_new_species(df_obs, last_days=30):
    """
    Obtiene las nuevas especies observadas en los últimos días.
    :param df_obs: DataFrame con las observaciones
    :param last_days: Número de días para considerar una especie como nueva
    :return: DataFrame con las nuevas especies
    """
    df_obs["observed_on"] = pd.to_datetime(df_obs["observed_on"], errors="coerce")

    df_species = df_obs[df_obs["taxon_rank"] == "species"]

    df_species_sorted = df_species.sort_values(by="observed_on")
    df_first = df_species_sorted.drop_duplicates(subset=["taxon_name"], keep="first")

    cutoff_date = datetime.today() - timedelta(days=last_days)
    
    df_new_species = df_first[df_first["observed_on"] > cutoff_date].reset_index(drop=True)

    # Mostrar la columna como fecha
    df_new_species["observed_on"] = df_new_species["observed_on"].dt.date

    return df_new_species

In [19]:
new_spe = get_new_species(df_obs)
new_spe

,id,created_at,updated_at,observed_on,observed_on_time,iconic_taxon,taxon_id,taxon_rank,taxon_name,latitude,...,identifiers,num_identification_agreements,num_identification_disagreements,device,kingdom,phylum,class,order,family,genus
0,432730,2025-04-09,2025-04-09,2025-04-07,07:58:00,aves,242811,species,Aegithalos caudatus,41.280418,...,"mediambient_ajelprat, badosa",1,0,web,Animalia,Chordata,Aves,Passeriformes,Aegithalidae,Aegithalos
1,432738,2025-04-09,2025-04-09,2025-04-07,08:29:00,plantae,252738,species,Papaver rhoeas,41.281011,...,"mediambient_ajelprat, xasalva",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Papaverales,Papaveraceae,Papaver
2,450048,2025-05-03,2025-05-05,2025-04-08,13:39:00,insecta,252648,species,Oxythyrea funesta,41.418766,...,"crismc, xasalva",1,0,web,Animalia,Arthropoda,Insecta,Coleoptera,Cetoniidae,Oxythyrea
3,450036,2025-05-03,2025-05-05,2025-04-25,12:02:00,plantae,245454,species,Echium plantagineum,41.416562,...,"crismc, loreto_rodriguez",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Lamiales,Boraginaceae,Echium
4,447661,2025-04-29,2025-05-01,2025-04-28,11:52:00,plantae,77959,species,Austrocylindropuntia cylindrica,41.265129,...,"xasalva, loreto_rodriguez",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Caryophyllales,Cactaceae,Austrocylindropuntia
5,447664,2025-04-29,2025-05-01,2025-04-28,11:54:00,plantae,264401,species,Marcus-kochia littorea,41.264936,...,"xasalva, loreto_rodriguez",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Capparales,Brassicaceae,Marcus-kochia
6,447668,2025-04-29,2025-05-01,2025-04-28,12:05:00,plantae,242353,species,Plantago coronopus,41.264936,...,"xasalva, loreto_rodriguez",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Plantaginales,Plantaginaceae,Plantago
7,447673,2025-04-29,2025-04-29,2025-04-28,12:11:00,plantae,250103,species,Olea europaea,41.265010,...,"xasalva, pauladelmar",1,0,web,Plantae,Tracheophyta,Magnoliopsida,Scrophulariales,Oleaceae,Olea
8,447658,2025-04-29,2025-04-30,2025-04-28,11:50:00,plantae,243911,species,Aloe vera,41.265104,...,"xasalva, bertinhaco",1,0,web,Plantae,Tracheophyta,Liliopsida,Asparagales,Xanthorrhoeaceae,Aloe
9,447639,2025-04-29,2025-05-01,2025-04-28,11:29:00,insecta,264823,species,Exhyalanthrax muscarius,41.264990,...,"xasalva, loreto_rodriguez",1,0,web,Animalia,Arthropoda,Insecta,Diptera,Bombyliidae,Exhyalanthrax


Función para sacar una foto de las nuevas especies

In [11]:
df_photos.head()

,id,photos_id,iconic_taxon,taxon_name,photos_medium_url,user_login,latitude,longitude,license_photo,attribution,path
0,450876,607443,actinopterygii,Mugil cephalus,https://minka-sdg.org/attachments/local_photos...,amb_platges,41.284844,2.098979,cc-by,"(c) AMB Platges, some rights reserved (CC BY)",450876_607443.jpg
1,450679,607033,plantae,Raphanus raphanistrum,https://minka-sdg.org/attachments/local_photos...,aasafa,41.402931,2.180586,cc-by,"(c) aasafa, some rights reserved (CC BY)",450679_607033.jpg
2,450678,607032,insecta,Aphis fabae,https://minka-sdg.org/attachments/local_photos...,aasafa,41.402931,2.180586,cc-by,"(c) aasafa, some rights reserved (CC BY)",450678_607032.jpg
3,450675,607029,mollusca,Ambigolimax valentianus,https://minka-sdg.org/attachments/local_photos...,aasafa,41.402972,2.180617,cc-by,"(c) aasafa, some rights reserved (CC BY)",450675_607029.jpg
4,450673,607026,reptilia,Tarentola mauritanica,https://minka-sdg.org/attachments/local_photos...,aasafa,41.402675,2.180239,cc-by,"(c) aasafa, some rights reserved (CC BY)",450673_607026.jpg


In [13]:
df_photos_new_species = df_photos[df_photos['id'].isin(new_spe['id'])]
df_photos_new_species

,id,photos_id,iconic_taxon,taxon_name,photos_medium_url,user_login,latitude,longitude,license_photo,attribution,path
7,450048,606122,insecta,Oxythyrea funesta,https://minka-sdg.org/attachments/local_photos...,crismc,41.418766,2.232517,cc-by,"(c) crismc, some rights reserved (CC BY)",450048_606122.jpg
15,450036,606103,plantae,Echium plantagineum,https://minka-sdg.org/attachments/local_photos...,crismc,41.416562,2.231530,cc-by,"(c) crismc, some rights reserved (CC BY)",450036_606103.jpg
16,450036,606104,plantae,Echium plantagineum,https://minka-sdg.org/attachments/local_photos...,crismc,41.416562,2.231530,cc-by,"(c) crismc, some rights reserved (CC BY)",450036_606104.jpg
71,447673,602814,plantae,Olea europaea,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265010,1.958497,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447673_602814.jpg
74,447668,602809,plantae,Plantago coronopus,https://minka-sdg.org/attachments/local_photos...,xasalva,41.264936,1.969611,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447668_602809.jpg
79,447664,602805,plantae,Marcus-kochia littorea,https://minka-sdg.org/attachments/local_photos...,xasalva,41.264936,1.969611,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447664_602805.jpg
78,447664,602804,plantae,Marcus-kochia littorea,https://minka-sdg.org/attachments/local_photos...,xasalva,41.264936,1.969611,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447664_602804.jpg
82,447661,602802,plantae,Austrocylindropuntia cylindrica,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265129,1.973101,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447661_602802.jpg
84,447659,602800,plantae,Opuntia stricta,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265104,1.974124,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447659_602800.jpg
85,447658,602799,plantae,Aloe vera,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265104,1.974124,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447658_602799.jpg


In [20]:
def download_photos(
    df_photos: pd.DataFrame, directorio: str = "minka_photos"
):
    """
    Function to download the photos resulting from the query.
    """
    # Create the folder, if it exists overwrite it
    if not os.path.exists(directorio):
        os.makedirs(directorio)

    session = requests.Session()

    # Iterate through the df_photos query result and download the photos in medium size
    for i, row in df_photos.iterrows():
        response = session.get(row["photos_medium_url"], stream=True)
        if response.status_code == 200:
            with open(f"{directorio}/{row['path']}", "wb") as out_file:
                out_file.write(response.content)
        del response

    # Even using .loc, we get a SettingWithCopyWarning message
    df_photos.loc[:, "abs_path"] = os.path.abspath(f"{directorio}/{df_photos['path']}")


In [21]:
def get_photos_new_species(df_new_species, df_photos):
    # El dataframe df_new_species tiene las especies nuevas, con el id de cada observación
    # Filtramos el dataframe de fotos para quedarnos solo con las fotos de las especies nuevas, las de los ids de esas observaciones.
    # Puedes utilizar el método isin() de pandas para filtrar el dataframe df_photos
    # df_photos['id'].isin(df_new_species['id'])
    # Investiga el método isin() y cómo se utiliza para filtrar un dataframe
    # Nos quedaríamos solo con una foto para cada especie nueva, así que podemos usar el método drop_duplicates() de pandas con subset(['id'])
    
    # Filtrar las fotos cuyas observaciones están en df_new_species
    df_photos_new_species = df_photos[df_photos['id'].isin(df_new_species['id'])]

    # Eliminar duplicados para quedarnos con una foto por observación
    df_photos_new_species = df_photos_new_species.drop_duplicates(subset=['id'])

    download_photos(df_photos_new_species)

    return df_photos_new_species

In [22]:
get_photos_new_species(new_spe, df_photos)

,id,photos_id,iconic_taxon,taxon_name,photos_medium_url,user_login,latitude,longitude,license_photo,attribution,path,abs_path
7,450048,606122,insecta,Oxythyrea funesta,https://minka-sdg.org/attachments/local_photos...,crismc,41.418766,2.232517,cc-by,"(c) crismc, some rights reserved (CC BY)",450048_606122.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
15,450036,606103,plantae,Echium plantagineum,https://minka-sdg.org/attachments/local_photos...,crismc,41.416562,2.231530,cc-by,"(c) crismc, some rights reserved (CC BY)",450036_606103.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
71,447673,602814,plantae,Olea europaea,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265010,1.958497,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447673_602814.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
74,447668,602809,plantae,Plantago coronopus,https://minka-sdg.org/attachments/local_photos...,xasalva,41.264936,1.969611,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447668_602809.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
79,447664,602805,plantae,Marcus-kochia littorea,https://minka-sdg.org/attachments/local_photos...,xasalva,41.264936,1.969611,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447664_602805.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
82,447661,602802,plantae,Austrocylindropuntia cylindrica,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265129,1.973101,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447661_602802.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
84,447659,602800,plantae,Opuntia stricta,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265104,1.974124,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447659_602800.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
85,447658,602799,plantae,Aloe vera,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265104,1.974124,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447658_602799.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
101,447639,602779,insecta,Exhyalanthrax muscarius,https://minka-sdg.org/attachments/local_photos...,xasalva,41.264990,1.980592,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447639_602779.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...
114,447590,602714,plantae,Carpobrotus acinaciformis,https://minka-sdg.org/attachments/local_photos...,xasalva,41.265039,1.985591,cc-by-nc,"(c) xavi salvador costa, some rights reserved ...",447590_602714.jpg,/home/anomalia/projects/Minka_Analysis/minka_p...


In [ ]:
df_new_species = get_new_species(df_obs, last_days=30)
df_new_species

Con esto estaríamos creando las funciones para extraer los datos. Luego estarían las de crear los gráficos y montar el informe.

# 5. Mapa calor para densidad de observaciones

Crear la función que toma un dataframe con el formato de df_obs (con esos nombres de columna, "latitude", "longitude") y lo mapee en el mapa de calor. Ese dataframe puede estar con las observaciones totales, filtrato por un kingdom, por un usuario, por un mes, o por lo que sea, pero no le afecta a la función, que lo hará siempre igual sobre un dataframe con las mismas columnas.

In [ ]:
# Fuente: https://stackoverflow.com/questions/53565979/export-a-folium-map-as-a-png
import io
from PIL import Image
# pip install selenium

def get_heatmap(df_obs, zoom_start:int):

    df_valid = df_obs.dropna(subset=["latitude", "longitude"])
    heat_data = df_valid[["latitude", "longitude"]].values.tolist()

    mean_lat = df_valid["latitude"].mean()
    mean_lon = df_valid["longitude"].mean()
   

    m = folium.Map(location=[mean_lat, mean_lon], zoom_start=zoom_start)
    HeatMap(heat_data).add_to(m)

    os.makedirs("figures", exist_ok=True)

    html_path = "figures/heatmap.html"
    m.save(html_path)

    img_data = m._to_png(3)
    img = Image.open(io.BytesIO(img_data))
    
    img.save('figures/heatmap_image.png')

In [4]:
get_heatmap(df_obs, 11)

Old Headless mode has been removed from the Chrome binary. Please use the new Headless mode (https://developer.chrome.com/docs/chromium/new-headless) or the chrome-headless-shell which is a standalone implementation of the old Headless mode (https://developer.chrome.com/blog/chrome-headless-shell).

